# Assignment Beta - Question 4

**Name:** Muhammad Sabeeh  
**Roll number:** 23K-0002  
**Course:** Agentic AI (AI4015)  
**Semester:** Fall 2026


# Assignment Beta - Question 4
Student roll: 23K-0002. Executed calculations; Wilson 95% intervals, sample standard deviation, unsmoothed sentence BLEU-4 and ROUGE F1.

In [1]:
import math, statistics, json
from pathlib import Path
ROOT=Path.cwd()
ROLL="23K-0002"
def classification(m):
    total=sum(map(sum,m)); n=[sum(r) for r in m]; pred=[sum(r[j] for r in m) for j in range(3)]
    correct=sum(m[i][i] for i in range(3)); acc=correct/total
    z=1.959963984540054
    center=(acc+z*z/(2*total))/(1+z*z/total)
    half=z*math.sqrt(acc*(1-acc)/total+z*z/(4*total*total))/(1+z*z/total)
    classes=[]
    for i,label in enumerate(['Positive','Negative','Neutral']):
        tp=m[i][i]; p=tp/pred[i]; r=tp/n[i]; f=2*tp/(n[i]+pred[i])
        classes.append({'class':label,'tp':tp,'fp':pred[i]-tp,'fn':n[i]-tp,'support':n[i],'precision':p,'recall':r,'f1':f})
    return {'matrix':m,'correct':correct,'n':total,'accuracy':acc,'wilson_95':[center-half,center+half],
            'classes':classes,'macro_f1':statistics.mean(c['f1'] for c in classes),
            'weighted_f1':sum(c['f1']*c['support'] for c in classes)/total,'micro_f1':acc}


def ngrams(tokens,n):
    from collections import Counter
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))


def text_metrics():
    ref='the delivery was late but the product quality is good'.split()
    cand='the product quality is good but delivery was slow'.split()
    precisions=[]; overlaps=[]; rouge={}
    for n in range(1,5):
        a=ngrams(ref,n); b=ngrams(cand,n); match=sum((a&b).values())
        precisions.append(match/sum(b.values())); overlaps.append(match)
        if n in [1,2]:
            p=match/sum(b.values()); r=match/sum(a.values())
            rouge[f'rouge_{n}']={'overlap':match,'precision':p,'recall':r,'f1':2*p*r/(p+r)}
    dp=[[0]*(len(cand)+1) for _ in range(len(ref)+1)]
    for i,a in enumerate(ref,1):
        for j,b in enumerate(cand,1):
            dp[i][j]=dp[i-1][j-1]+1 if a==b else max(dp[i-1][j],dp[i][j-1])
    lcs=dp[-1][-1]; p=lcs/len(cand); r=lcs/len(ref)
    rouge['rouge_l']={'lcs':lcs,'precision':p,'recall':r,'f1':2*p*r/(p+r)}
    bp=math.exp(min(0,1-len(ref)/len(cand)))
    probs=[.60,.40,.50,.80,.70,.30,.25,.50,.20]
    exact_matches=[(0,5),(1,6),(2,7),(3,8),(4,9),(5,4),(6,1),(7,2)]
    chunks=1+sum(b[0]!=a[0]+1 or b[1]!=a[1]+1 for a,b in zip(exact_matches,exact_matches[1:]))
    mp=8/9; mr=8/10; harmonic=10*mp*mr/(mr+9*mp)
    exact_meteor=harmonic*(1-.5*(chunks/8)**3)
    return {'reference':ref,'candidate':cand,'overlaps_1_to_4':overlaps,'modified_precisions':precisions,
            'brevity_penalty':bp,'bleu4':bp*math.exp(sum(map(math.log,precisions))/4),
            **rouge,'meteor_exact_only':exact_meteor,'meteor_exact_chunks':chunks,
            'token_probabilities':probs,'sum_log_probabilities':sum(map(math.log,probs)),
            'mean_negative_log_probability':-statistics.mean(map(math.log,probs)),
            'perplexity':math.exp(-statistics.mean(map(math.log,probs)))}


def q4():
    a=classification([[100,5,15],[8,80,12],[20,10,50]])
    b=classification([[106,4,10],[6,84,10],[9,7,64]])
    runs=[.847,.840,.853,.837,.850]
    return {'stage1':a,'stage2':b,'accuracy_delta':b['accuracy']-a['accuracy'],
            'prompt_runs':runs,'prompt_mean':statistics.mean(runs),'prompt_sample_sd':statistics.stdev(runs),
            'prompt_population_sd':statistics.pstdev(runs),'text':text_metrics()}


## (1)-(5) Classification metrics
Rows are gold labels. Precision=TP/(TP+FP); recall=TP/(TP+FN); F1=2TP/(2TP+FP+FN). Macro is unweighted; weighted uses supports 120/100/80; micro equals accuracy for this single-label task.

In [2]:
r=q4()
for s in ["stage1","stage2"]:
    print(s,json.dumps(r[s],indent=2))
print("Absolute accuracy improvement:",r["accuracy_delta"])

stage1 {
  "matrix": [
    [
      100,
      5,
      15
    ],
    [
      8,
      80,
      12
    ],
    [
      20,
      10,
      50
    ]
  ],
  "correct": 230,
  "n": 300,
  "accuracy": 0.7666666666666667,
  "wilson_95": [
    0.715618653578636,
    0.8109717617682728
  ],
  "classes": [
    {
      "class": "Positive",
      "tp": 100,
      "fp": 28,
      "fn": 20,
      "support": 120,
      "precision": 0.78125,
      "recall": 0.8333333333333334,
      "f1": 0.8064516129032258
    },
    {
      "class": "Negative",
      "tp": 80,
      "fp": 15,
      "fn": 20,
      "support": 100,
      "precision": 0.8421052631578947,
      "recall": 0.8,
      "f1": 0.8205128205128205
    },
    {
      "class": "Neutral",
      "tp": 50,
      "fp": 27,
      "fn": 30,
      "support": 80,
      "precision": 0.6493506493506493,
      "recall": 0.625,
      "f1": 0.6369426751592356
    }
  ],
  "macro_f1": 0.7546357028584273,
  "weighted_f1": 0.7659362987080266,
  "micro_f1": 0.76

## (6) Ordering stability
Report the sample standard deviation with divisor n-1, because these five orderings sample the possible orderings.

In [3]:
print("Mean:",r["prompt_mean"],"Sample SD:",r["prompt_sample_sd"],"Population SD:",r["prompt_population_sd"])

Mean: 0.8453999999999999 Sample SD: 0.006730527468185542 Population SD: 0.00601996677731697


## (7)-(8) Text quality and perplexity
Tokenization is lower-case whitespace. The reference has 10 words and the candidate has 9. BLEU is unsmoothed; ROUGE includes precision, recall and F1. METEOR uses NLTK default α=.9, β=3, γ=.5; exact alignment yields 8 matches in 3 chunks and neither stemming nor WordNet adds a match.

In [4]:
print(json.dumps(r["text"],indent=2))

{
  "reference": [
    "the",
    "delivery",
    "was",
    "late",
    "but",
    "the",
    "product",
    "quality",
    "is",
    "good"
  ],
  "candidate": [
    "the",
    "product",
    "quality",
    "is",
    "good",
    "but",
    "delivery",
    "was",
    "slow"
  ],
  "overlaps_1_to_4": [
    8,
    5,
    3,
    2
  ],
  "modified_precisions": [
    0.8888888888888888,
    0.625,
    0.42857142857142855,
    0.3333333333333333
  ],
  "brevity_penalty": 0.8948393168143697,
  "bleu4": 0.474954953300978,
  "rouge_1": {
    "overlap": 8,
    "precision": 0.8888888888888888,
    "recall": 0.8,
    "f1": 0.8421052631578948
  },
  "rouge_2": {
    "overlap": 5,
    "precision": 0.625,
    "recall": 0.5555555555555556,
    "f1": 0.5882352941176471
  },
  "rouge_l": {
    "lcs": 5,
    "precision": 0.5555555555555556,
    "recall": 0.5,
    "f1": 0.5263157894736842
  },
  "meteor_exact_only": 0.7867739898989898,
  "meteor_exact_chunks": 3,
  "token_probabilities": [
    0.6,
    

## Optional independent NLTK check
If NLTK and WordNet are installed, run the following separately. It is not needed for the standard-library calculations.
```python
from nltk.translate.meteor_score import meteor_score
meteor_score([r["text"]["reference"]], r["text"]["candidate"])
# 0.78677398989899
```
Stage 1 particularly misses Neutral. Stage 2 improves all class F1 scores, but using the same 300 reviews for prompt development and assessment can overstate generalization. Paired significance cannot be reconstructed from these two aggregate matrices.